In [ ]:
! pip install azureml-sdk

In [ ]:
import os
import json
import requests

from azureml.core import Workspace
from azureml.core.model import Model
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice, Webservice

In [ ]:
config_file_path = "/content/azure_config.json"

with open(config_file_path, 'r') as file:
    data = json.load(file)

subscription_id = data["subscription_id"]
resource_group = data["resource_group"]
workspace_name = data["workspace_name"]
region = data["region"]

In [ ]:
try:
    ws = Workspace(subscription_id=subscription_id, resource_group=resource_group, workspace_name=workspace_name)
    print(f'Workspace {workspace_name} found.')
except Exception as e:
    ws = Workspace.create(name=workspace_name,
                          subscription_id=subscription_id,
                          resource_group=resource_group,
                          location=region)
    print(f'Workspace {workspace_name} created.')


In [ ]:
model_path = '/content/models/RRDB_ESRGAN_x4.pth'
model_name='image-super-resolution'

In [ ]:
registered_models = Model.list(workspace=ws)
model_already_registered = any(model.name == model_name for model in registered_models)

if model_already_registered:
    print(f"Model '{model_name}' already exists in the workspace.")
else:
    registered_model = Model.register(model_path=model_path, model_name=model_name, workspace=ws)
    print(f"Model '{model_name}' registered successfully.")

In [ ]:
conda_env = Environment('my-conda-env')

conda_packages = [
    'python=3.8',
    'pytorch',
    'torchvision',
    'numpy',
    'pillow'
]

conda_deps = CondaDependencies.create(conda_packages=conda_packages)

conda_env.python.conda_dependencies = conda_deps

In [ ]:
inference_config = InferenceConfig(source_directory='src', entry_script='score.py', environment=conda_env)

In [ ]:
aci_config = AciWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)

In [ ]:
existing_services = Webservice.list(workspace=ws)
service_already_exists = any(service.name == 'image-super-resolution' for service in existing_services)

if service_already_exists:
    existing_service = Webservice(workspace=ws, name='image-super-resolution')
    existing_service.delete()
    print("Existing service 'image-super-resolution' deleted.")

service = Model.deploy(workspace=ws,
                       name='image-super-resolution',
                       models=[registered_model],
                       inference_config=inference_config,
                       deployment_config=aci_config)
service.wait_for_deployment(show_output=True)
print("Service 'image-super-resolution' deployed successfully.")

In [ ]:
# print(service.get_logs())

In [ ]:
scoring_uri = service.scoring_uri